# Defining Cofactors

This cookbook demonstrates how to define cofactors (any small molecules that are not the bound ligand) with 1) the ``openfe`` CLI during the planning phase and 2) directly using the Python API.

Similar to `ligand` definition, `cofactors` can be loaded from in SDF file format, and multiple molecules can be defined in a single file.

## 1) CLI: define cofactors during ``openfe plan-rbfe-network``

When using openfe's CLI, including cofactors in an RBFE calculation is as simple as passing in the ``-C`` argument with the path to the SDF file containing cofactors:

```bash
openfe plan-rbfe-network -M ligands.sdf -p protein.pdb -C cofactors.sdf
```

## 2) Python API: define cofactors in the ``ChemicalSystem``s 

If using the ``openfe`` Python API to set up an RBFE network, cofactors need to be included when constructing the ``ChemicalSystem``s.

### Define the components (protein, ligands, cofactors)

Define the ligands, protein, and solvent in the same way as a typical RBFE calculation. 


<div class="alert alert-warning">
Note that partial charges are already assigned for both the ligands and cofactors. See <a href="https://docs.openfree.energy/en/latest/cookbook/user_charges.html">Assigning partial charges to an OpenFF Molecule</a>, but note that this parametrizes cofactors using the small molecule force field, which may not be applicable in all situations.
</div>


In [ ]:
import openfe
from rdkit import Chem
from openff.toolkit import Molecule

# Create a list of OpenFF molecules
offmols = [
    Molecule.from_rdkit(m)
    for m in Chem.SDMolSupplier("assets/cofactors/eg5_ligands_charged.sdf", removeHs=False)
]

# Now we convert them to SmallMoleculeComponent
ligands = [openfe.SmallMoleculeComponent.from_openff(mol) for mol in offmols]

# defaults are water with NaCl at 0.15 M
solvent = openfe.SolventComponent()

# load in the EG5 protein
protein = openfe.ProteinComponent.from_pdb_file("assets/cofactors/eg5_protein.pdb")

Load the cofactor(s) to be included in the simulations:

In [3]:
# load in the cofactor from eg5_cofactors.sdf, here we know that there is only one cofactor in the file
# We also go through OpenFF to assign partial charges
offmol = Molecule.from_rdkit(Chem.MolFromMolFile("assets/cofactors/eg5_cofactor_charged.sdf", removeHs=False))
cofactor = openfe.SmallMoleculeComponent.from_openff(offmol)
cofactor

SmallMoleculeComponent(name=3L9H)

### Create the Ligand Network

(This doesn't require any specific cofactor steps, but is included for completeness)

In [4]:
mapper = openfe.LomapAtomMapper(max3d=1.0, element_change=False, threed=True)
scorer = openfe.lomap_scorers.default_lomap_score
network_planner = openfe.ligand_network_planning.generate_minimal_spanning_network

ligand_network = network_planner(
    ligands=ligands,
    mappers=mapper,
    scorer=scorer
)

Mapping:  28%|##8       | 107/378 [00:01<00:03, 71.05it/s]

### Define an RBFE Protocol

(This doesn't require any specific cofactor steps, but is included for completeness)

In [5]:
from openfe.protocols.openmm_rfe import RelativeHybridTopologyProtocol

default_settings = RelativeHybridTopologyProtocol.default_settings()
protocol = RelativeHybridTopologyProtocol(default_settings)

### Create an ``AlchemicalNetwork`` that includes cofactors

When creating the legs of``AlchemicalNetwork``, you must explicitly define the cofactors as part of the complex legs' ``ChemicalSystem``s.

In [6]:
transformations = []
for mapping in ligand_network.edges:
    for leg in ['solvent', 'complex']:
        # use the solvent and protein created above
        sysA_dict = {'ligand': mapping.componentA,
                     'solvent': solvent}
        sysB_dict = {'ligand': mapping.componentB,
                     'solvent': solvent}
        
        if leg == 'complex':
            sysA_dict['protein'] = protein
            sysB_dict['protein'] = protein
            # Add cofactors to both sysA and sysB 
            sysA_dict['cofactor'] = cofactor
            sysB_dict['cofactor'] = cofactor 
        
        # we don't have to name objects, but it can make things (like filenames) more convenient
        sysA = openfe.ChemicalSystem(sysA_dict, name=f"{mapping.componentA.name}_{leg}")
        sysB = openfe.ChemicalSystem(sysB_dict, name=f"{mapping.componentB.name}_{leg}")
        
        prefix = "rbfe_"  # prefix is only to exactly reproduce CLI
        
        transformation = openfe.Transformation(
            stateA=sysA,
            stateB=sysB,
            mapping=mapping,
            protocol=protocol,  # use protocol created above
            name=f"{prefix}{sysA.name}_{sysB.name}"
        )
        transformations.append(transformation)

network = openfe.AlchemicalNetwork(transformations)

Cofactors can now be seen in the complex nodes of the alchemical network:

In [7]:
network.nodes

frozenset({ChemicalSystem(name=lig_CHEMBL1096002_solvent, components={'ligand': SmallMoleculeComponent(name=lig_CHEMBL1096002), 'solvent': SolventComponent(name=O, Na+, Cl-)}),
           ChemicalSystem(name=lig_CHEMBL1083517_complex, components={'ligand': SmallMoleculeComponent(name=lig_CHEMBL1083517), 'solvent': SolventComponent(name=O, Na+, Cl-), 'protein': ProteinComponent(name=), 'cofactor': SmallMoleculeComponent(name=3L9H)}),
           ChemicalSystem(name=lig_CHEMBL1096003_complex, components={'ligand': SmallMoleculeComponent(name=lig_CHEMBL1096003), 'solvent': SolventComponent(name=O, Na+, Cl-), 'protein': ProteinComponent(name=), 'cofactor': SmallMoleculeComponent(name=3L9H)}),
           ChemicalSystem(name=lig_CHEMBL1085692_complex, components={'ligand': SmallMoleculeComponent(name=lig_CHEMBL1085692), 'solvent': SolventComponent(name=O, Na+, Cl-), 'protein': ProteinComponent(name=), 'cofactor': SmallMoleculeComponent(name=3L9H)}),
           ChemicalSystem(name=lig_CHEMBL10